In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

px.defaults.template = "plotly_white"

In [2]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"
pio.renderers.default = "iframe"

In [3]:
df = pd.read_excel("./data/online_retail_II.xlsx")

In [4]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df = df.dropna(subset=['Customer ID'])

df['Revenue'] = df['Quantity'] * df['Price']
df['IsReturn'] = df['Quantity'] < 0

In [5]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,IsReturn
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,False


## 📊 RFM Segmentation

In [6]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'Invoice': 'nunique',
    'Revenue': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# Interactive scatter
fig = px.scatter(
    rfm,
    x="Recency",
    y="Monetary",
    size="Frequency",
    color="Frequency",
    hover_data=["CustomerID"],
    title="Customer Segmentation (RFM)",
)

fig.show()

## 📉 Customer Activity Over Time

In [7]:
monthly_active = (
    df.groupby(pd.Grouper(key='InvoiceDate', freq='M'))['Customer ID']
    .nunique()
    .reset_index(name='ActiveCustomers')
)

fig = px.line(
    monthly_active,
    x='InvoiceDate',
    y='ActiveCustomers',
    title="Monthly Active Customers",
)

fig.show()

/tmp/ipykernel_198647/3673799247.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df.groupby(pd.Grouper(key='InvoiceDate', freq='M'))['Customer ID']


## 🛍️ Top Products

In [8]:
products = df.groupby('Description').agg({
    'Revenue': 'sum',
    'Quantity': 'sum'
}).reset_index()

top_products = products.sort_values('Revenue', ascending=False).head(20)

fig = px.bar(
    top_products,
    x='Revenue',
    y='Description',
    orientation='h',
    title="Top Products by Revenue",
)

fig.show()

## 🌍 Country-Level Revenue

In [9]:
country = df.groupby('Country')['Revenue'].sum().reset_index()

fig = px.choropleth(
    country,
    locations='Country',
    locationmode='country names',
    color='Revenue',
    title="Revenue by Country",
)

fig.show()

/tmp/ipykernel_198647/2411211032.py:3: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


## 🔄 Cohort Retention Heatmap

In [10]:
df = df.copy()

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df = df.dropna(subset=['Customer ID'])

df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)

df['CohortMonth'] = (
    df.groupby('Customer ID')['InvoiceDate']
    .transform('min')
    .dt.to_period('M')
    .astype(str)
)

cohort = (
    df.groupby(['CohortMonth', 'InvoiceMonth'])['Customer ID']
    .nunique()
    .reset_index()
)

cohort['CohortMonth'] = pd.to_datetime(cohort['CohortMonth'])
cohort['InvoiceMonth'] = pd.to_datetime(cohort['InvoiceMonth'])

cohort['PeriodIndex'] = (
    (cohort['InvoiceMonth'].dt.year - cohort['CohortMonth'].dt.year) * 12 +
    (cohort['InvoiceMonth'].dt.month - cohort['CohortMonth'].dt.month)
)

cohort_pivot = cohort.pivot_table(
    index='CohortMonth',
    columns='PeriodIndex',
    values='Customer ID'
)

retention = cohort_pivot.divide(cohort_pivot.iloc[:, 0], axis=0)

retention = retention.fillna(0)

In [11]:
import plotly.express as px

fig = px.imshow(
    retention,
    aspect='auto',
    color_continuous_scale='Blues',
    labels=dict(x="Months Since First Purchase", y="Cohort", color="Retention"),
    title="Customer Retention Cohort (Churn View)"
)

fig.update_layout(
    xaxis=dict(side="top"),
    yaxis=dict(tickformat="%Y-%m")
)

fig.show()